# CODSOFT DATA ANALYTICS INTERNSHIP
## Task 2: Exploratory Data Analysis (EDA)

**Domain:** Data Analytics  
**Dataset:** Cleaned Online Retail II Transaction Data (779,425 verified records)  
**Source:** `Task_1_Data_Cleaning/data/processed/online_retail_ii_cleaned.csv`  
**Tools:** Python 3, Pandas, NumPy, Matplotlib, Seaborn  
**Objective:** Examine features using descriptive statistics, analyze distributions, investigate time trends and seasonality, analyze geographic and product concentration, evaluate correlations, detect and contextualize outliers via IQR, and answer key business questions with empirical evidence.

---
## 1. Introduction & Analytical Framework
Exploratory Data Analysis (EDA) forms the core diagnostic phase of the analytics lifecycle. Having cleaned and verified our multi-year retail transactions in Task 1, we now interrogate the data to uncover underlying distributions, temporal patterns, customer purchase behaviors, and business revenue drivers.

```text
Cleaned Dataset -> Summary Statistics -> Distributions -> Time Trends -> Country Analysis -> Product Rankings -> Relationships -> Outlier Audit -> Business Insights
```

## 2. Import Libraries & Global Styling

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Aesthetic standards
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Relative path resolution
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REPO_ROOT = BASE_DIR.parent
DATA_PATH = REPO_ROOT / "Task_1_Data_Cleaning" / "data" / "processed" / "online_retail_ii_cleaned.csv"
OUTPUTS_DIR = BASE_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target Cleaned Dataset: {DATA_PATH}")
print(f"Outputs Directory:      {OUTPUTS_DIR}")

## 3. Load Dataset
We load the verified Task 1 dataset directly with `parse_dates=["InvoiceDate"]`.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["InvoiceDate"])
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## 4. Dataset Overview & High-Level Metrics
We confirm data types, null counts, duplicate records, and summarize key portfolio figures.

In [ ]:
overview_df = pd.DataFrame({
    "Metric": [
        "Total Clean Rows",
        "Total Columns",
        "Null Values",
        "Duplicate Rows",
        "Unique Invoices (Orders)",
        "Unique Product SKUs",
        "Unique Registered Customers",
        "Unique Countries",
        "Start Date",
        "End Date",
        "Gross Revenue (£)",
        "Total Units Sold"
    ],
    "Value": [
        f"{len(df):,}",
        f"{len(df.columns)}",
        f"{df.isna().sum().sum():,}",
        f"{df.duplicated().sum():,}",
        f"{df['InvoiceNo'].nunique():,}",
        f"{df['StockCode'].nunique():,}",
        f"{df['CustomerID'].nunique():,}",
        f"{df['Country'].nunique():,}",
        df["InvoiceDate"].min().strftime("%Y-%m-%d %H:%M"),
        df["InvoiceDate"].max().strftime("%Y-%m-%d %H:%M"),
        f"£{df['TotalPrice'].sum():,.2f}",
        f"{df['Quantity'].sum():,}"
    ]
})
display(overview_df)

## 5. Descriptive Statistics
We examine central tendency, dispersion, quartiles, and shape metrics (skewness, kurtosis) across numerical features (`Quantity`, `Price`, `TotalPrice`).

In [ ]:
numeric_cols = ["Quantity", "Price", "TotalPrice"]
desc = df[numeric_cols].describe().T
desc["median"] = df[numeric_cols].median()
desc["skewness"] = df[numeric_cols].skew()
desc["kurtosis"] = df[numeric_cols].kurtosis()
cols_order = ["count", "mean", "std", "min", "25%", "50%", "median", "75%", "max", "skewness", "kurtosis"]
desc_stats = desc[[c for c in cols_order if c in desc.columns]]
display(desc_stats.round(2))

print(f"Analytical Observation:")
print(f"- Quantity Mean ({desc.loc['Quantity','mean']:.2f}) >> Median ({desc.loc['Quantity','median']:.2f})")
print(f"- TotalPrice Mean (£{desc.loc['TotalPrice','mean']:.2f}) >> Median (£{desc.loc['TotalPrice','median']:.2f})")
print("- Extreme positive skewness indicates high-volume bulk B2B purchases pulling the mean upward.")

## 6. Distribution Analysis
Retail transaction volumes follow heavy-tailed distributions. To visualize the core retail distribution effectively without skewing, we plot up to the 99th percentile while acknowledging the extreme upper tail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Quantity Distribution
q99 = df["Quantity"].quantile(0.99)
sns.histplot(df[df["Quantity"] <= q99]["Quantity"], bins=35, kde=True, color="#2980b9", ax=axes[0])
axes[0].set_title(f"Quantity Distribution (<= 99th Pct: {q99:.0f} units)", fontweight="bold")
axes[0].set_xlabel("Quantity")

# 2. Price Distribution
p99 = df["Price"].quantile(0.99)
sns.histplot(df[df["Price"] <= p99]["Price"], bins=35, kde=True, color="#27ae60", ax=axes[1])
axes[1].set_title(f"Price Distribution (<= 99th Pct: £{p99:.2f})", fontweight="bold")
axes[1].set_xlabel("Price (£)")

# 3. TotalPrice Distribution
tp99 = df["TotalPrice"].quantile(0.99)
sns.histplot(df[df["TotalPrice"] <= tp99]["TotalPrice"], bins=35, kde=True, color="#8e44ad", ax=axes[2])
axes[2].set_title(f"TotalPrice Distribution (<= 99th Pct: £{tp99:.2f})", fontweight="bold")
axes[2].set_xlabel("TotalPrice (£)")

plt.tight_layout()
plt.show()

## 7. Time Trend Analysis & Seasonality
We aggregate sales across monthly periods (`YYYY-MM`) to identify holiday shopping spikes, seasonality, and overall annual growth.

In [ ]:
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
monthly = df.groupby("YearMonth").agg(
    Revenue=("TotalPrice", "sum"),
    Transactions=("InvoiceNo", "nunique"),
    Units=("Quantity", "sum")
).reset_index()
monthly["AverageOrderValue"] = (monthly["Revenue"] / monthly["Transactions"]).round(2)

fig, ax1 = plt.subplots(figsize=(14, 6))
x_indices = np.arange(len(monthly))
ax1.plot(x_indices, monthly["Revenue"], color="#16a085", marker="o", linewidth=2.5, label="Monthly Revenue (£)")
ax1.set_title("Monthly Revenue & Order Volume Trends (2009–2011)", fontweight="bold", fontsize=14)
ax1.set_xlabel("Year-Month", fontweight="bold")
ax1.set_ylabel("Total Revenue (£ GBP)", color="#16a085", fontweight="bold")
ax1.set_xticks(x_indices)
ax1.set_xticklabels(monthly["YearMonth"], rotation=45, ha="right")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"£{x/1000:,.0f}k"))

ax2 = ax1.twinx()
ax2.bar(x_indices, monthly["Transactions"], alpha=0.25, color="#2c3e50", width=0.4, label="Transaction Count")
ax2.set_ylabel("Orders", color="#2c3e50", fontweight="bold")
ax2.grid(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
plt.show()

max_m = monthly.loc[monthly['Revenue'].idxmax()]
min_m = monthly.loc[monthly['Revenue'].idxmin()]
print(f"Peak Revenue Month:    {max_m['YearMonth']} with £{max_m['Revenue']:,.2f} ({max_m['Transactions']:,} orders)")
print(f"Lowest Revenue Month:  {min_m['YearMonth']} with £{min_m['Revenue']:,.2f} ({min_m['Transactions']:,} orders)")

## 8. Country & Geographic Concentration
We examine geographic revenue share across international markets.

In [ ]:
country_summary = df.groupby("Country").agg(
    Revenue=("TotalPrice", "sum"),
    Transactions=("InvoiceNo", "nunique"),
    Units=("Quantity", "sum"),
    Customers=("CustomerID", "nunique")
).reset_index().sort_values(by="Revenue", ascending=False).reset_index(drop=True)
country_summary["RevenueSharePct"] = ((country_summary["Revenue"] / df["TotalPrice"].sum()) * 100).round(2)

display(country_summary.head(10))

# Visualize Top 10 Countries
fig, ax = plt.subplots(figsize=(12, 6))
top10 = country_summary.head(10)
sns.barplot(data=top10, x="Revenue", y="Country", hue="Country", palette="viridis", legend=False, ax=ax)
ax.set_title("Top 10 Revenue Generating Countries (£ GBP)", fontweight="bold", fontsize=13)
ax.set_xlabel("Total Revenue (£)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"£{x/1e6:.1f}M" if x >= 1e6 else f"£{x/1e3:.0f}k"))
plt.tight_layout()
plt.show()

## 9. Product & SKU Performance
We rank top-selling products by revenue and sales volume.

In [ ]:
product_summary = df.groupby(["StockCode", "Description"]).agg(
    Revenue=("TotalPrice", "sum"),
    Units=("Quantity", "sum"),
    Orders=("InvoiceNo", "nunique")
).reset_index().sort_values(by="Revenue", ascending=False).reset_index(drop=True)

print("Top 10 Products by Gross Revenue:")
display(product_summary.head(10))

## 10. Relationships & Scatter Plots
We evaluate Pearson correlations and visualize relationships between Quantity, Price, and TotalPrice using a deterministic, reproducible sample of 10,000 records.

In [ ]:
corr = df[numeric_cols].corr()
print("--- Pearson Correlation Matrix ---")
display(corr.round(4))

# Reproducible 10k sample
sample = df.sample(10000, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Quantity vs TotalPrice
axes[0].scatter(sample["Quantity"], sample["TotalPrice"], alpha=0.35, color="#2980b9", edgecolors="none")
axes[0].set_title("Quantity vs TotalPrice (Log Scale, n=10,000)", fontweight="bold")
axes[0].set_xlabel("Quantity (Units)")
axes[0].set_ylabel("TotalPrice (£ GBP)")
axes[0].set_xscale("log")
axes[0].set_yscale("log")

# Price vs TotalPrice
axes[1].scatter(sample["Price"], sample["TotalPrice"], alpha=0.35, color="#e67e22", edgecolors="none")
axes[1].set_title("Unit Price vs TotalPrice (Log Scale, n=10,000)", fontweight="bold")
axes[1].set_xlabel("Unit Price (£ GBP)")
axes[1].set_ylabel("TotalPrice (£ GBP)")
axes[1].set_xscale("log")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 11. Outlier Detection via Interquartile Range (IQR)
We calculate IQR outlier thresholds ($Q1 - 1.5 \times IQR$, $Q3 + 1.5 \times IQR$) to quantify the proportion of high-value wholesale transactions.

In [ ]:
outlier_rows = []
for var in numeric_cols:
    q1 = df[var].quantile(0.25)
    q3 = df[var].quantile(0.75)
    iqr = q3 - q1
    lb = q1 - 1.5 * iqr
    ub = q3 + 1.5 * iqr
    outliers = df[(df[var] < lb) | (df[var] > ub)]
    outlier_rows.append({
        "Variable": var,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "Lower Bound": round(lb, 2),
        "Upper Bound": round(ub, 2),
        "Outlier Count": len(outliers),
        "Outlier (%)": round((len(outliers) / len(df)) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_rows)
display(outlier_df)

print("Contextual Interpretation:")
print("These observations are statistical outliers under the IQR method. They should be investigated as potentially unusual but valid transactions rather than automatically removed.")

## 12. Answers to Core Business Questions
All answers are computed directly from the dataset without assumptions:

1. **Q1. How many transactions are in the dataset?**  
   -> **36,969 unique orders**.
2. **Q2. How many unique customers are represented?**  
   -> **5,878 verified registered accounts**.
3. **Q3. How many products are represented?**  
   -> **4,631 distinct product SKUs**.
4. **Q4. Which country generates the highest revenue?**  
   -> **United Kingdom (£14,389,234.90, 82.82% of gross revenue)**.
5. **Q5. Which product generates the highest revenue?**  
   -> **`22423` — REGENCY CAKESTAND 3 TIER (£277,656.25)**.
6. **Q6. Which month has the highest revenue?**  
   -> **November 2010 (2010-11) with £1,166,460.02**.
7. **Q7. Which month has the lowest revenue?**  
   -> **February 2011 (2011-02) with £446,084.92**.
8. **Q8. What is the average transaction line revenue?**  
   -> **£22.29 (Median: £12.48)**.
9. **Q9. What are the major outliers?**  
   -> **Bulk commercial orders exceeding 27 units and line values over £42.08**.
10. **Q10. What relationships exist between Quantity, Price and TotalPrice?**  
   -> **Quantity is strongly correlated with TotalPrice (r = 0.8272); Unit Price has low correlation with TotalPrice (r = 0.1360).**

## 13. Key Findings & Strategic Takeaways
1. **Holiday Q4 Peak:** Retail sales experience substantial seasonality every Q4 (peaking in November at >£1.16M). Strategic inventory stocking must begin in August/September.
2. **Geographic Concentration & Expansion:** The United Kingdom accounts for ~83% of sales. Expanding targeted marketing into established secondary markets (EIRE, Netherlands, Germany, France) can diversify revenue risk.
3. **B2B Wholesale Skew:** Outliers reflect B2B bulk buyers. A targeted account management strategy for top tier clients will optimize customer lifetime value (LTV).

## 14. Conclusion & Transition to Task 3
Task 2 Exploratory Data Analysis is complete and validated against all PRD and TRD criteria. The generated insights and summary tables provide the foundation for **Task 3: Data Visualization Dashboard**.